# Chapter 11 — Organizing Files

Source: [Automate the Boring Stuff with Python, 3rd Ed. — Chapter 11](https://automatetheboringstuff.com/3e/chapter11.html)

**Setup note:** Uses `shutil`, `os`, and `zipfile` from the standard library.

This notebook favors *reading* over *writing*: each section has a short explanation followed by real, runnable code. Read it, run it, tweak a value and re-run it — that's the fastest way to build reading fluency.

---


## 11.1 Copying, moving, and deleting with shutil

The `shutil` module handles whole-file operations that plain `open()` can't: copying (`copy`, `copytree`), moving/renaming (`move`), and deleting (`rmtree` for whole folders). Permanent deletion is risky — the third-party `send2trash` module sends files to the recycle bin/trash instead.


In [1]:
import shutil
from pathlib import Path

src = Path('shutil_demo_src.txt')
src.write_text('demo content')
dst = Path('shutil_demo_copy.txt')
shutil.copy(src, dst)
print(dst.exists(), dst.read_text())

src.unlink()
dst.unlink()


True demo content


## 11.2 Walking a directory tree

`os.walk(top)` recursively visits a folder and every subfolder, yielding `(dirpath, dirnames, filenames)` for each one — the standard way to process an entire folder tree, e.g. to find all `.txt` files.


In [2]:
import os
from pathlib import Path

demo_root = Path('walk_demo')
(demo_root / 'sub').mkdir(parents=True, exist_ok=True)
(demo_root / 'a.txt').write_text('a')
(demo_root / 'sub' / 'b.txt').write_text('b')

for dirpath, dirnames, filenames in os.walk(demo_root):
    for name in filenames:
        print(Path(dirpath) / name)

import shutil
shutil.rmtree(demo_root)  # clean up


walk_demo/a.txt
walk_demo/sub/b.txt


## 11.3 Pattern-matching filenames with glob

`Path.glob(pattern)` finds files matching a wildcard pattern (`*` for any characters) without writing a manual loop — handy for "all CSVs in this folder" style tasks.


In [3]:
from pathlib import Path

demo = Path('glob_demo')
demo.mkdir(exist_ok=True)
for name in ['a.txt', 'b.txt', 'c.csv']:
    (demo / name).write_text('x')

txt_files = list(demo.glob('*.txt'))
print(txt_files)

import shutil
shutil.rmtree(demo)


[PosixPath('glob_demo/b.txt'), PosixPath('glob_demo/a.txt')]


## 11.4 Compressing and extracting with zipfile

The `zipfile` module reads and writes `.zip` archives without shelling out to an external program: `ZipFile(name, 'w')` to create/add files, `ZipFile(name, 'r')` plus `.extractall()` to unpack.


In [4]:
import zipfile
from pathlib import Path

Path('zip_demo.txt').write_text('hello zip')
with zipfile.ZipFile('demo.zip', 'w') as zf:
    zf.write('zip_demo.txt')

with zipfile.ZipFile('demo.zip', 'r') as zf:
    print(zf.namelist())

Path('zip_demo.txt').unlink()
Path('demo.zip').unlink()


['zip_demo.txt']


## 11.5 Safer deletion with send2trash

`os.unlink()` and `shutil.rmtree()` (from earlier in this chapter) delete permanently — there's no undo. The third-party `send2trash` module's `send2trash(path)` instead moves the file/folder to the OS's actual Trash/Recycle Bin, exactly like deleting it through the file manager — so a mistake is still recoverable. The book recommends using it by default for any delete operation in a real automation script, reserving the permanent `os` functions for cases where you're certain.

**Environment note:** this sandbox's filesystem doesn't have a real desktop trash can to move files into (it's headless, and this particular mount also blocks the permission check `send2trash` needs), so the cell below catches that and explains what would happen on a normal desktop OS instead.

In [5]:
from pathlib import Path
from send2trash import send2trash

demo_file = Path('send2trash_demo.txt')
demo_file.write_text('temporary file')

try:
    send2trash(str(demo_file))
    print('Sent to trash. Still exists on disk?', demo_file.exists())
except Exception as e:
    print(f'{type(e).__name__} in this sandbox (expected, no desktop Trash available here):', e)
    print('On a real desktop OS, this call recoverably trashes the file instead of deleting it outright.')
    demo_file.unlink()   # clean up manually since send2trash could not


TrashPermissionError in this sandbox (expected, no desktop Trash available here): [Errno 13] Permission denied: ''
On a real desktop OS, this call recoverably trashes the file instead of deleting it outright.
